# 06 · Candidatos y decisión

Ejecutar todas las celdas en orden. Datos históricos y simulaciones educativas; no representan una aplicación financiera real.

In [1]:
from pathlib import Path
import sys, os
os.environ["EVIDENTLY_DO_NOT_TRACK"]="1"
base=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"_config.yml").exists())
project=base/"actividad_3/proyecto_inicial"
sys.path.insert(0,str(project/"src"))
os.environ["BANK_ROOT"]=str(project)
import numpy as np, pandas as pd
from bank_ops.data import load, partitions
from bank_ops.config import FEATURES, SEED
from bank_ops.model import pipeline, metrics
data=load(); train,validation,test=partitions(data)


## Comparar sobre la misma validación
El bosque no tiene garantizado mejorar. Este notebook muestra resultados; implementar el gate es parte de la actividad.

In [2]:
results={}
for kind in ["baseline","candidate"]:
    estimator=pipeline(kind);estimator.fit(train[FEATURES],(train.y=="yes").astype(int))
    results[kind]=metrics((validation.y=="yes").astype(int),estimator.predict_proba(validation[FEATURES])[:,1])
display(pd.DataFrame(results).T[["average_precision","recall","brier","roc_auc"]])

,average_precision,recall,brier,roc_auc
baseline,0.168429,0.0,0.104156,0.624764
candidate,0.155108,0.0,0.106439,0.585485


## Política
AP no cae más de 0.005, recall no más de 0.02 y Brier no sube más de 0.01. Elegibilidad no sustituye revisión de equidad, latencia y contexto.

In [3]:
b=results["baseline"];c=results["candidate"]
display(pd.DataFrame({"metrica":["AP","Recall","Brier"],"cambio":[c["average_precision"]-b["average_precision"],c["recall"]-b["recall"],c["brier"]-b["brier"]]}))

,metrica,cambio
0,AP,-0.013321
1,Recall,0.000000
2,Brier,0.002283


## Ensayo de recuperación
Completar gate, seguir laboratorio de réplica y rollback, comprobar versión HTTP tras reinicio. El notebook no cambia el modelo activo.